This is the notebook for cleaning up the conversation records recorded in NALA-Assess Postgres database (which are exported as `data_dump.sql` not in this repository to protect confidentiality of users)

In [ ]:
import re
import pandas as pd

In [ ]:
# Define cleaning functions
def remove_emojis(text):
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F700-\U0001F77F"  # alchemical symbols
        u"\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
        u"\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
        u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        u"\U0001FA00-\U0001FA6F"  # Chess Symbols
        u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        u"\U00002702-\U000027B0"  # Dingbats
        u"\U000024C2-\U0001F251" 
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def remove_formatting(text):
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)  # Bold
    text = re.sub(r'__(.*?)__', r'\1', text)      # Underline
    text = re.sub(r'\*(.*?)\*', r'\1', text)      # Italic
    text = re.sub(r'<.*?>', '', text)            # HTML tags
    return text

def clean_text(text):
    text = remove_emojis(text)
    text = text.replace('\n', ' ').replace('\r', ' ')  # Remove newlines
    text = remove_formatting(text)
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

In [ ]:
answers_data = []
with open('data_dump.sql', 'r', encoding='utf-8') as file:
    inside_copy = False
    columns = []
    for line in file:
        if line.startswith('COPY public.answers'):
            # Extract column names
            columns = re.search(r'\((.*?)\)', line).group(1).split(', ')
            inside_copy = True
            continue
        if inside_copy:
            if line.startswith('\\.'):
                inside_copy = False
                continue
            # Process data rows
            row = line.strip().split('\t')  # Tab-delimited in SQL dumps
            answers_data.append([clean_text(value) for value in row])

# Save to CSV
answers_df = pd.DataFrame(answers_data, columns=columns)
answers_df.to_csv('NALA-Assess/cleaned_answers.csv', index=False, encoding='utf-8-sig')

In [ ]:
questions_data = []
with open('data_dump.sql', 'r', encoding='utf-8') as file:
    inside_copy = False
    columns = []
    for line in file:
        if line.startswith('COPY public.questions'):
            # Extract column names
            columns = re.search(r'\((.*?)\)', line).group(1).split(', ')
            inside_copy = True
            continue
        if inside_copy:
            if line.startswith('\\.'):
                inside_copy = False
                continue
            # Process data rows
            row = line.strip().split('\t')  # Tab-delimited in SQL dumps
            questions_data.append([clean_text(value) for value in row])

# Save to CSV
questions_df = pd.DataFrame(questions_data, columns=columns)
questions_df.to_csv('NALA-Assess/cleaned_questions.csv', index=False, encoding='utf-8-sig')

In [ ]:
messages_data = []
with open('data_dump.sql', 'r', encoding='utf-8') as file:
    inside_copy = False
    columns = []
    for line in file:
        if line.startswith('COPY public.messages'):
            # Extract column names
            columns = re.search(r'\((.*?)\)', line).group(1).split(', ')
            inside_copy = True
            continue
        if inside_copy:
            if line.startswith('\\.'):
                inside_copy = False
                continue
            # Process data rows
            row = line.strip().split('\t')  # Tab-delimited in SQL dumps
            messages_data.append([clean_text(value) for value in row])

# Save to CSV
messages_df = pd.DataFrame(messages_data, columns=columns)
messages_df.to_csv('NALA-Assess/cleaned_messages.csv', index=False, encoding='utf-8-sig')

In [ ]:
# process conversations data
conversations_data = []
with open('data_dump.sql', 'r', encoding='utf-8') as file:
    inside_copy = False
    columns = []
    for line in file:
        if line.startswith('COPY public.conversations'):
            # Extract column names
            columns = re.search(r'\((.*?)\)', line).group(1).split(', ')
            inside_copy = True
            continue
        if inside_copy:
            if line.startswith('\\.'):
                inside_copy = False
                continue
            # Process data rows
            row = line.strip().split('\t')  # Tab-delimited in SQL dumps
            conversations_data.append([clean_text(value) for value in row])
# Save to CSV
conversations_df = pd.DataFrame(conversations_data, columns=columns)
conversations_df.to_csv('NALA-Assess/cleaned_conversations.csv', index=False, encoding='utf-8-sig')

In [ ]:
users_data = []
with open('data_dump.sql', 'r', encoding='utf-8') as file:
    inside_copy = False
    columns = []
    for line in file:
        if line.startswith('COPY public.users'):
            # Extract column names
            columns = re.search(r'\((.*?)\)', line).group(1).split(', ')
            inside_copy = True
            continue
        if inside_copy:
            if line.startswith('\\.'):
                inside_copy = False
                continue
            # Process data rows
            row = line.strip().split('\t')  # Tab-delimited in SQL dumps
            users_data.append([clean_text(value) for value in row])
# Save to CSV
users_df = pd.DataFrame(users_data, columns=columns)
users_df.to_csv('NALA-Assess/cleaned_users.csv', index=False, encoding='utf-8-sig')

In [ ]:
questions_topics_data = []
with open('data_dump.sql', 'r', encoding='utf-8') as file:
    inside_copy = False
    columns = []
    for line in file:
        if line.startswith('COPY public.question_topics'):
            columns = re.search(r'\((.*?)\)', line).group(1).split(', ')
            inside_copy = True
            continue
        if inside_copy:
            if line.startswith('\\.'):
                inside_copy = False
                continue
            row = line.strip().split('\t')
            questions_topics_data.append([clean_text(value) for value in row])
questions_topics_df = pd.DataFrame(questions_topics_data, columns=columns)

In [ ]:
# check for any null values in the dataframes
# NO NULL VALUES FOUND IN ANY OF THE DATAFRAMES
print("Null values in Answers:", answers_df.isnull().sum())
print("Null values in Questions:", questions_df.isnull().sum())
print("Null values in Messages:", messages_df.isnull().sum())
print("Null values in Conversations:", conversations_df.isnull().sum())
print("Null values in Users:", users_df.isnull().sum())
print("Null values in Questions_Topics:", questions_topics_df.isnull().sum())

In [ ]:
# count the number of rows in each df
print(f"Answers: {len(answers_df)} rows")
print(f"Questions: {len(questions_df)} rows")
print(f"Messages: {len(messages_df)} rows")
print(f"Conversations: {len(conversations_df)} rows")
print(f"Users: {len(users_df)} rows")
print(f"Questions_Topics: {len(questions_topics_df)} rows")

--> number of rows have been checked against postgres db rowcount ✅

In [ ]:
# find number of unique 'id' in each df
print(f"Unique question ids in answers_df: {len(answers_df['id'].unique())}")
print(f"Unique question ids in questions_df: {len(questions_df['id'].unique())}")
print(f"Unique message ids in messages_df: {len(messages_df['id'].unique())}")
print(f"Unique conversation ids in conversations_df: {len(conversations_df['id'].unique())}")
print(f"Unique user ids in users_df: {len(users_df['id'].unique())}")
print(f"Unique question ids in questions_topics_df: {len(questions_topics_df['question_id'].unique())}")

In [ ]:
# find what are the extra question ids in questions_df that are not in questions_topics_df
extra_question_ids = set(questions_df['id'].unique()) - set(questions_topics_df['question_id'].unique())
print(f"Number of extra question ids in questions_df: {len(extra_question_ids)}")
print(f"Extra question ids in questions_df: {extra_question_ids}")

# locate the extra question ids in questions_df
extra_questions = questions_df[questions_df['id'].isin(extra_question_ids)]
extra_questions

In [ ]:
# find message content for the message ids in extra_questions
extra_message_ids = extra_questions['message_id'].unique()
extra_messages = messages_df[messages_df['id'].isin(extra_message_ids)]
extra_messages

**Note**:

After classifying the questions separately using LLM given the topic context:
- "Q1. what is wronskian sums" --> NOT a relevant question so no topic/subtopic ids
- "Q2. List the degrees of freedom ($N_f$) formula and identify its variables." --> topic id 2 (message id 5179, question id 1235)
- "Q3. Compare the Final Value Theorem (FVT) and the Initial Value Theorem (IVT) --> topic id 3 (message id 6717, question id 1612)
- "Q4. What is the Initial Value Theorem (IVT)?" --> topic id 3 (message id 9042, question id 2156)
- "Q5. If shaft work is negligible." --> NOT a question but more of a follow-up

i will put this into question topics, maybe not question subtopics since we just need the number of topics for calculation of reimbursement.

In [ ]:
# add question ids to the question_topics_df
new_question_topics = [
    {'question_id': 1235, 'topic_id': 2},
    {'question_id': 1612, 'topic_id': 3},
    {'question_id': 2156, 'topic_id': 3}
]

new_question_topics_df = pd.DataFrame(new_question_topics)
questions_topics_df = pd.concat([questions_topics_df, new_question_topics_df], ignore_index=True)

questions_topics_df.tail()

In [ ]:
print(len(questions_topics_df))
# save new questions_topics_df to csv
questions_topics_df.to_csv('NALA-Assess/cleaned_questions_topics.csv', index=False, encoding='utf-8-sig')

In [ ]:
# print updated unique question ids in questions_topics_df
print(questions_topics_df['question_id'].nunique())

start merging the dfs

In [ ]:
# Merge users with conversations and drop duplicate id column from conversations
merged_df = pd.merge(conversations_df, users_df, left_on='user_id', right_on='id', how='inner')
merged_df.drop(columns=['id_y'], inplace=True)
merged_df

Remove the last_accesed column from the `conversations` table, we need to use the timestamp column from the `messages` table.

In [ ]:
# rename id_x to conversation_id
merged_df.rename(columns={'id_x': 'conversation_id'}, inplace=True)
# drop last_accessed column
merged_df.drop(columns=['last_accessed'], inplace=True)
merged_df

In [ ]:
# Merge conversations with messages
merged_df = pd.merge(merged_df, messages_df, left_on='conversation_id', right_on='conversation_id', how='inner')
merged_df

In [ ]:
merged_df.rename(columns={'id': 'message_id', 'content': 'message_content', '"timestamp"': 'timestamp'}, inplace=True)
merged_df

In [ ]:
# merge messages with questions
# so all the rows in the merged df should be questions only
merged_df = pd.merge(merged_df, questions_df, left_on='message_id', right_on='message_id', how='inner')
merged_df

In [ ]:
# check that the sender is only user
merged_df['sender'].unique()

In [ ]:
# check for no null values in the merged dataframe
merged_df.info()

In [ ]:
# check for how many answered and unanswered questions
print("Answered questions:", merged_df[merged_df['status'] == 'ANSWERED'].shape[0])
print("Unanswered questions:", merged_df[merged_df['status'] == 'AWAITING_ANSWER'].shape[0])

In [ ]:
# rename columns from questions
merged_df.rename(columns={
    'id': 'question_id',
    'solo_taxonomy_level': 'question_solo_level',
    'reasoning': 'solo_level_reasoning',
    'grade': 'question_grade',
    'status': 'question_status'
}, inplace=True)

# drop created at and updated at columns
merged_df.drop(columns=['created_at', 'updated_at'], inplace=True)

merged_df

In [ ]:
# check if question ids are duplicated in the merged df
# no duplicates
print(f"Number of unique question ids in merged df: {len(merged_df['question_id'].unique())}")

In [ ]:
questions_topics_df.info()

In [ ]:
merged_df.info()

In [ ]:
# change question_id to same str type in question_topics_df
questions_topics_df['question_id'] = questions_topics_df['question_id'].astype(str)
questions_topics_df.info()

In [ ]:
# Aggregate topics
question_topics = questions_topics_df.groupby('question_id')['topic_id'].apply(list).reset_index()
print(f"len of unique question_topics: {len(question_topics)}")

there should be 2596 unique question ids in the merged df

In [ ]:
question_topics

In [ ]:
# check if all question ids in merged df are in question_topics
merged_question_ids = set(merged_df['question_id'].unique())
question_topics_ids = set(question_topics['question_id'].unique())
missing_question_ids = merged_question_ids - question_topics_ids
print(f"Missing question ids in question_topics: {missing_question_ids}")

In [ ]:
# merge the question_topics with the merged df on question_id
final_df = pd.merge(merged_df, question_topics, left_on='question_id', right_on='question_id', how='inner')
# rename topic_id to topic_ids
final_df.rename(columns={'topic_id': 'topic_ids'}, inplace=True)
final_df

In [ ]:
# convert timestamp to sgt explicitly
final_df['timestamp_sgt'] = pd.to_datetime(final_df['timestamp']).dt.tz_convert('Asia/Singapore')
# drop timestamp column
final_df.drop(columns=['timestamp'], inplace=True)
final_df

In [ ]:
# rename 'title' to 'conversation_title'
final_df.rename(columns={'title': 'conversation_title'}, inplace=True)
# reorder columns to have user_id, conversation_id, conversation_title, message_id, question_id, sender, message_content, question_solo_level, solo_level_reasoning, question_grade, question_status, topic_ids
final_df = final_df[[
    'user_id', 'conversation_id', 'conversation_title', 'message_id', 'question_id',
    'sender', 'message_content', 'question_solo_level', 'solo_level_reasoning',
    'question_grade', 'question_status', 'topic_ids', 'timestamp_sgt'
]]
final_df

In [ ]:
# sort final_df by user_id, conversation_id and within each conversation sort by timestamp_sgt
final_df.sort_values(by=['user_id', 'conversation_id', 'timestamp_sgt'], inplace=True)
final_df

In [ ]:
# remove rows with user id 1 and 2
final_df = final_df[~final_df['user_id'].isin(['1', '2'])]
final_df

In [ ]:
# find what is the earliest timestamp
print("Earliest timestamp in final_df:", final_df['timestamp_sgt'].min())
# find what is the latest timestamp
print("Latest timestamp in final_df:", final_df['timestamp_sgt'].max())

dataset is correctly within the time period of the study.

In [ ]:
# checking for null values and data types
final_df.info()

In [ ]:
# save final df to csv
final_df.to_csv('NALA-Assess/questions_cleaned_data.csv', index=False, encoding='utf-8-sig')

In [ ]:
# group by user_id and conversation_id and count the number of messages in each conversation
conversation_counts = final_df.groupby(['user_id', 'conversation_id']).size().reset_index(name='question_count')
conversation_counts

In [ ]:
# check if there are any prestructural questions in the final df
prestructural_questions = final_df[final_df['question_solo_level'] == 'Prestructural']
print(f"Number of prestructural questions: {len(prestructural_questions)}")